# Phase 2 Chemistry Classifier Diagnostics

This notebook loads the Phase 2 artifacts generated by `scripts/train_chemistry.py` and inspects held-out predictions, feature importance, calibration, and representative voltage/dQ-dV examples.

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from triagenet.config import DATA_PROCESSED, MODELS_DIR, REPO_ROOT
from triagenet.features.eol_cycle import CHEMISTRY_FEATURES, extract_features
from triagenet.features.ic_curve import compute_ic_curve

report_dir = REPO_ROOT / 'reports' / 'phase2_chemistry'
model = joblib.load(MODELS_DIR / 'chemistry_xgb.joblib')
metadata = json.loads((MODELS_DIR / 'chemistry_metadata.json').read_text())
predictions = pd.read_csv(report_dir / 'shape_leave_one_cell_predictions.csv')
cycles = pd.read_parquet(DATA_PROCESSED / 'cycles.parquet')
metadata['metrics']

In [ ]:
display(predictions.groupby(['chemistry', 'predicted_chemistry']).size().unstack(fill_value=0))
display(pd.Series(metadata['feature_importances']).sort_values(ascending=False).head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, image_name, title in zip(
    axes,
    ['feature_importance.png', 'shape_leave_one_cell_out_calibration.png', 'shape_without_soh_window_leave_one_cell_out_calibration.png'],
    ['Feature importance', 'Matched-SOH calibration', 'Full-SOH baseline calibration'],
):
    ax.imshow(plt.imread(report_dir / image_name))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

In [ ]:
examples = []
correct = predictions[predictions['chemistry'] == predictions['predicted_chemistry']].head(1)
incorrect = predictions[predictions['chemistry'] != predictions['predicted_chemistry']].head(1)
examples.extend(correct.to_dict('records'))
examples.extend(incorrect.to_dict('records'))

for example in examples:
    row = cycles[(cycles.cell_id == example['cell_id']) & (cycles.cycle_index == example['cycle_index'])].iloc[0]
    features = pd.DataFrame([extract_features(row)])[list(CHEMISTRY_FEATURES)]
    voltage_grid, ic_curve = compute_ic_curve(row.voltage_curve, row.current_curve, row.time_curve_s)
    print(example['cell_id'], 'cycle', example['cycle_index'], 'true=', example['chemistry'], 'pred=', example['predicted_chemistry'])
    print(model.top_features(features.iloc[0], k=5))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(row.time_curve_s, row.voltage_curve)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Voltage (V)')
    axes[0].set_title('Discharge voltage')
    axes[1].plot(voltage_grid, ic_curve)
    axes[1].set_xlabel('Voltage (V)')
    axes[1].set_ylabel('dQ/dV')
    axes[1].set_title('Incremental capacity')
    plt.tight_layout()
    plt.show()

## What I Observed

The matched-SOH leave-one-cell-out result is near ceiling, but the harder leave-one-dataset-out split is not trainable yet because CALCE contains only LCO and MIT contains only LFP. Voltage-level and plateau-shape features dominate the current classifier; dQ/dV features are present but not leading on this two-chemistry dataset. The remaining risk for Phase 3 is source confounding: chemistry and dataset are still perfectly aligned until Sandia or another cross-chemistry source is added.

## Voltage-leakage audit (Phase 2.5)

The Phase 2 top features were mostly absolute-voltage statistics. This audit compares that baseline with the production shape-feature model and probes robustness when held-out test voltage curves are shifted by additive offsets.

In [ ]:
robustness = json.loads((report_dir / 'voltage_robustness.json').read_text())
display(pd.DataFrame(robustness['results']).rename_axis('shift_v'))
plt.figure(figsize=(7, 4))
plt.imshow(plt.imread(report_dir / 'voltage_robustness.png'))
plt.axis('off')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex='col')
for row_idx, chemistry in enumerate(['LCO', 'LFP']):
    cycle = cycles[cycles.chemistry == chemistry].iloc[0]
    voltage = pd.Series(cycle.voltage_curve, dtype=float)
    v_norm = (voltage - voltage.min()) / (voltage.max() - voltage.min())
    axes[row_idx, 0].plot(cycle.time_curve_s, voltage)
    axes[row_idx, 0].set_ylabel(f'{chemistry}\nVoltage (V)')
    axes[row_idx, 1].plot(cycle.time_curve_s, v_norm)
    axes[row_idx, 1].set_ylabel('Normalized V')
axes[0, 0].set_title('Raw voltage')
axes[0, 1].set_title('Per-cycle normalized voltage')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 1].set_xlabel('Time (s)')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, image_name, title in zip(
    axes,
    ['feature_importance.png', 'voltage_axis_feature_importance.png'],
    ['Shape model feature importance', 'Absolute-voltage baseline importance'],
):
    ax.imshow(plt.imread(report_dir / image_name))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

The shape model is invariant to additive voltage offsets in this probe, while the absolute-voltage baseline loses accuracy under a -0.2 V test-time shift. The result is not a full substitute for cross-source validation: CALCE is still the only LCO source and MIT is still the only LFP source, and the shape model also uses C-rate features that may encode protocol. For production, this audit is a guardrail against voltage-axis shortcuts, not proof that all deployment shifts are solved.